# **비정형 문서에서 Knowledge Graph 자동 생성**

## SimpleKGPipeline을 활용한 Entity Extraction 및 Knowledge Graph 구축

---

## 학습 목표
- SimpleKGPipeline을 사용한 비정형 문서 → Knowledge Graph 자동 변환 이해
- 엔티티 및 관계 추출을 위한 LLM 기반 프롬프트 커스터마이징
- 스키마 정의 방법 (EntityType, RelationType, potential_schema) 습득
- Entity Resolution을 통한 중복 엔티티 병합 기법 이해
- 생성된 Lexical Graph 구조 (Document → Chunk → Entity 관계) 파악

## 사전 준비

### 선수 학습
- PRJ04_W3_001: Neo4j 소개
- PRJ04_W3_002: LangChain 통합
- PRJ04_W3_003: Knowledge Graph 구축

### 필수 지식
- Python 기본 문법
- Neo4j 기본 개념 및 Cypher 쿼리
- LLM API 기본 이해 (OpenAI, Anthropic, Ollama)

### 환경 요구사항
- Neo4j AuraDB 계정 및 연결 정보 (또는 Neo4j Desktop)
- Neo4j APOC core 라이브러리 설치 필요 (AuraDB는 기본 포함)
- 필요한 패키지:
  ```bash
  pip install neo4j-graphrag langchain-neo4j langchain-openai langchain-google-genai python-dotenv
  ```

---

## **1. 개념 소개**

> 💡 **이 노트북에서 배우는 것을 한 줄로**
>
> **"뉴스 기사 한 편 → `await pipeline.run_async()` 한 줄 → 자동으로 Neo4j KG 완성"** 
> 지금까지는 수동으로 Cypher 문을 작성해 노드·관계를 만들었지만, 이번 장에서는 
> `SimpleKGPipeline` 이 **LLM 을 활용해 비정형 텍스트를 KG 로 자동 변환**하는 과정을 익힙니다.


### **1.1 SimpleKGPipeline이란?**

**SimpleKGPipeline**은 `neo4j-graphrag` 라이브러리의 핵심 도구로, 비정형 텍스트 데이터에서 자동으로 Knowledge Graph를 생성합니다.

#### 주요 특징
- **LLM 기반 자동 추출**: 대규모 언어 모델을 활용하여 엔티티와 관계를 자동으로 식별
- **스키마 정의**: 추출할 엔티티 타입과 관계 타입을 미리 정의
- **Entity Resolution**: 동일한 엔티티를 자동으로 병합
- **Lexical Graph 생성**: Document → Chunk → Entity 구조로 출처 추적 가능

#### 처리 흐름
```
비정형 텍스트/PDF
    ↓
Text Splitting (청크 분할)
    ↓
LLM Entity/Relation Extraction (엔티티 및 관계 추출)
    ↓
Entity Resolution (중복 병합)
    ↓
Neo4j Knowledge Graph 저장
```

> 💡 **SimpleKGPipeline 내부 동작을 한눈에**
>
```mermaid
flowchart LR
    T[📄 비정형 텍스트<br/>뉴스·보고서] --> SPL[Text Splitter<br/>FixedSize/Recursive]
    SPL --> LLM1[LLM Entity 추출<br/>사전 정의 스키마 기반]
    LLM1 --> LLM2[LLM Relation 추출<br/>엔티티 쌍 사이 관계]
    LLM2 --> ER[Entity Resolution<br/>동일인물 병합]
    ER --> EMB[Chunk Embedding<br/>OpenAI / LocalEmbedder]
    EMB --> WR[Neo4j Writer<br/>Document-Chunk-Entity]
    WR --> KG[(🗺️ Knowledge Graph)]
    style LLM1 fill:#fff4e1
    style LLM2 fill:#fff4e1
    style ER fill:#e8f5e9
```

> 🔑 **SimpleKGPipeline 의 핵심 가치**: 수작업으로 하던 **"Partitioning → NER → Relation Extraction → Resolution → Ingestion"** 을 한 줄의 `await pipeline.run_async(text=...)` 로 끝내준다.


### **1.2 생성되는 그래프 구조**

SimpleKGPipeline은 다음과 같은 **Lexical Graph** 구조를 생성합니다:

```
(:Document {path: "news.txt"})
    ↑
    │[:FROM_DOCUMENT]
    │
(:Chunk {text: "...", embedding: [...], index: 0})
    ↑
    │[:NEXT_CHUNK]          ← 인접 청크끼리 연결
    │
(:Chunk {index: 1}) ... 
    ↑
    │[:FROM_CHUNK]
    │
(:Person {name: "젠슨 황"}) ─[:WORKS_FOR]→ (:Company {name: "엔비디아"})
(:Technology {name: "GPU"})
```

#### 주요 노드 타입
- `Document`: 원본 문서
- `Chunk`: 텍스트 청크 (embedding 벡터 포함)
- 커스텀 엔티티: 스키마에 정의된 엔티티 (Person, Company, Technology 등)
- 엔티티 노드는 공통 라벨 `__Entity__` 도 함께 가짐

#### 주요 관계 타입
- `FROM_DOCUMENT`: **Chunk → Document** (청크가 어느 문서에서 왔는지, 방향 주의)
- `NEXT_CHUNK`: Chunk → Chunk (같은 문서 내 인접 청크 연결)
- `FROM_CHUNK`: Entity → Chunk (엔티티가 어느 청크에서 추출되었는지)
- 커스텀 관계: 스키마에 정의된 관계 (WORKS_FOR, DEVELOPS 등)

> ⚠️ **주의**: `neo4j-graphrag` 의 기본 설정은 `Chunk-[:FROM_DOCUMENT]->Document` 입니다. 과거 예제에서 보던 `Document-[:HAS_CHUNK]->Chunk` 와 방향이 반대이니 Cypher 쿼리 작성 시 주의하세요.

---

## **2. 환경 설정**

### **2.1 필수 라이브러리 임포트**

In [1]:
# 표준 라이브러리
import os
from dotenv import load_dotenv

# Neo4j 및 GraphRAG
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import FixedSizeSplitter
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

# LangChain (검증 및 조회용)
from langchain_neo4j import Neo4jGraph

print("✓ 라이브러리 임포트 완료")

/Users/seungiljang/workspace/004_prj/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ 라이브러리 임포트 완료


### **2.2 환경 변수 로드 및 연결**

In [2]:
# 환경 변수 로드
load_dotenv()

# Neo4j 연결 정보
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

# API 키 확인
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다."
assert NEO4J_URI, "NEO4J_URI가 설정되지 않았습니다."

print("✓ 환경 변수 로드 완료")

✓ 환경 변수 로드 완료


In [3]:
# Neo4j 드라이버 초기화 (SimpleKGPipeline용)
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

# LangChain Neo4jGraph (검증용)
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

# 연결 테스트
result = graph.query("RETURN 'Connected!' as message")
print("Neo4j 연결 상태:", result[0]['message'])

Neo4j 연결 상태: Connected!


### **2.2.1 드라이버 재연결 유틸리티**

노트북을 반복 실행하다 보면 끝부분 `driver.close()` 나 커널 타임아웃으로 드라이버가 닫혀 `DriverError: Driver closed` 가 날 수 있습니다. 아래 유틸리티를 정의해 두고, 이후 셀에서 필요 시 호출하세요.

In [4]:
def ensure_neo4j_connection():
    """driver / graph 가 닫혀 있으면 재생성"""
    global driver, graph
    need_new_driver = getattr(driver, "_closed", False)
    if need_new_driver:
        print("⚠️ driver 가 닫혀 있어 재연결합니다.")
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    try:
        driver.verify_connectivity()
    except Exception as e:
        print(f"⚠️ 재연결 시도 — 원인: {e}")
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
        driver.verify_connectivity()
    # LangChain Neo4jGraph 도 재생성
    try:
        graph.query("RETURN 1")
    except Exception:
        graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)
    print("✓ Neo4j 연결 상태 정상")

ensure_neo4j_connection()

✓ Neo4j 연결 상태 정상


### **2.3 기존 Knowledge Graph 정리 (필수)**

이전 노트북(W3_001~004)에서 만들어진 **UNIQUE 제약조건**과 노드들이 Neo4j에 남아 있으면, SimpleKGPipeline 이 같은 이름의 엔티티(예: `Technology {name:'HBM'}`)를 추가하려 할 때 `ConstraintValidationFailed` 오류가 발생합니다.

> 📌 **왜 MERGE 가 아니라 CREATE?**  
> `neo4j-graphrag` 의 `Neo4jWriter` 는 내부적으로 `CREATE (n:__KGBuilder__ {__tmp_internal_id: ...}) SET n:Technology` 순서로 노드를 쌓고, 나중에 Entity Resolution 단계에서 이름으로 병합합니다. 이 때문에 기존 유니크 제약이 있으면 `SET n:Technology` 단계에서 충돌이 납니다.

아래 셀은 **모든 노드·관계·제약·인덱스를 삭제**합니다. 실습 전용 DB(AuraDB Free 등)에서만 실행하세요.

In [5]:
# ⚠️ 실습용 DB 초기화: 기존 제약조건 + 인덱스 + 데이터 모두 삭제
# W3_001~004 에서 만든 technology_name_unique 등 UNIQUE 제약이 남아 있으면
# SimpleKGPipeline 실행 시 ConstraintValidationFailed 오류가 발생합니다.

# 1) 모든 노드와 관계 삭제
graph.query("MATCH (n) DETACH DELETE n")

# 2) 모든 제약조건 삭제
constraints = graph.query("SHOW CONSTRAINTS YIELD name")
for row in constraints:
    graph.query(f"DROP CONSTRAINT `{row['name']}` IF EXISTS")

# 3) 남아 있는 인덱스(제약 소유가 아닌 것) 삭제 — 선택이지만 깔끔하게 비우기 위해 수행
indexes = graph.query("SHOW INDEXES YIELD name, owningConstraint WHERE owningConstraint IS NULL RETURN name")
for row in indexes:
    graph.query(f"DROP INDEX `{row['name']}` IF EXISTS")

# 4) 검증
remaining_nodes = graph.query("MATCH (n) RETURN count(n) AS cnt")[0]['cnt']
remaining_constraints = len(graph.query("SHOW CONSTRAINTS YIELD name"))
remaining_indexes = len(graph.query("SHOW INDEXES YIELD name"))

print(f"✓ DB 초기화 완료 — 노드 {remaining_nodes}개 / 제약 {remaining_constraints}개 / 인덱스 {remaining_indexes}개")


✓ DB 초기화 완료 — 노드 0개 / 제약 0개 / 인덱스 0개


---

## **3. LLM 및 Embedder 설정**

### **3.1 다양한 LLM Provider 설정**

SimpleKGPipeline은 다양한 LLM Provider를 지원합니다. 여기서는 OpenAI를 사용합니다.

In [6]:
# OpenAI LLM 설정
llm = OpenAILLM(
    model_name="gpt-4.1-mini",  
    model_params={
        "temperature": 0,  # 결정론적 출력을 위해 0으로 설정
        "response_format": {"type": "json_object"},  # JSON 출력 강제
        "max_tokens": 4000,
    }
)

print("✓ LLM 설정 완료:", llm.model_name)

✓ LLM 설정 완료: gpt-4.1-mini


#### 💡 참고: 다른 LLM Provider 사용 예시

```python
# Anthropic (Claude)
from neo4j_graphrag.llm import AnthropicLLM
llm = AnthropicLLM(
    model_name="claude-sonnet-4-20250514",
    model_params={"temperature": 0, "max_tokens": 4000}
)

# Ollama (로컬)
from neo4j_graphrag.llm import OllamaLLM
llm = OllamaLLM(
    model_name="llama3:8b",
    model_params={"temperature": 0}
)
```

### **3.2 Embedder 설정**

In [7]:
# OpenAI Embeddings 설정 (청크 벡터화용)
embedder = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 또는 "text-embedding-3-large"
)

print("✓ Embedder 설정 완료")

✓ Embedder 설정 완료


---

## **4. 스키마 정의**

### **4.1 기본 스키마 (문자열 방식)**

가장 간단한 스키마 정의 방법으로, 엔티티 타입과 관계 타입을 문자열 리스트로 지정합니다.

> 💡 **"스키마를 줄까, 자유롭게 맡길까?"**
>
```mermaid
graph LR
    subgraph A[스키마 기반 추출]
      A1[엔티티 타입<br/>Company/Person/Technology]
      A2[관계 타입<br/>DEVELOPS/INVESTS_IN]
      A3[potential_schema<br/>허용 조합 제한]
    end
    subgraph B[자유 추출 no schema]
      B1[LLM 이 임의로 라벨링]
      B2[노드·관계 타입이 폭발적으로 증가]
    end
    A --> R1[✅ 일관성 · 쿼리 쉬움]
    B --> R2[⚠️ 탐색 초기엔 편하지만<br/>프로덕션 쿼리 작성 어려움]
```

> 🔑 **실전 권장**: 
> 
> 1단계 — 스키마 없이 작은 문서 1~2편으로 **자유 추출** 해 어떤 개념이 나오는지 관찰. 
> 
> 2단계 — 관찰 결과를 바탕으로 `entities` / `relations` / `potential_schema` 를 정의. 
> 
> 3단계 — 본격 적재 시 이 **명시 스키마** 로 run_async 실행. 


In [9]:
# 추출할 엔티티 타입 정의
entities = [
    "Person",        # 사람 (CEO, 연구자 등)
    "Company",       # 회사
    "Technology",    # 기술 (GPU, AI 등)
    "Product"        # 제품
]

# 추출할 관계 타입 정의
relations = [
    "WORKS_FOR",      # 사람 → 회사
    "DEVELOPS",       # 회사 → 기술/제품
    "COMPETES_WITH",  # 회사 ↔ 회사
    "USES"            # 제품 → 기술
]

# 허용되는 관계 패턴 (선택사항이지만 권장)
potential_schema = [
    ("Person", "WORKS_FOR", "Company"),
    ("Company", "DEVELOPS", "Technology"),
    ("Company", "DEVELOPS", "Product"),
    ("Company", "COMPETES_WITH", "Company"),
    ("Product", "USES", "Technology")
]

print("✓ 기본 스키마 정의 완료")
print(f"  - 엔티티 타입: {len(entities)}개")
print(f"  - 관계 타입: {len(relations)}개")
print(f"  - 허용 패턴: {len(potential_schema)}개")

✓ 기본 스키마 정의 완료
  - 엔티티 타입: 4개
  - 관계 타입: 4개
  - 허용 패턴: 5개


### **4.2 상세 스키마 (속성 포함)**

엔티티와 관계에 추가 설명과 속성을 정의하여 LLM이 더 정확하게 추출할 수 있도록 합니다.

In [9]:
# 속성을 포함한 상세 스키마
entities_detailed = [
    "Person",
    {"label": "Company", "description": "회사 또는 조직"},
    {
        "label": "Product",
        "properties": [
            {"name": "price", "type": "FLOAT"},
            {"name": "release_date", "type": "DATE"}
        ]
    },
    "Technology"
]

relations_detailed = [
    "WORKS_FOR",
    {
        "label": "DEVELOPS",
        "description": "회사가 제품이나 기술을 개발하는 관계"
    },
    {
        "label": "COMPETES_WITH",
        "properties": [
            {"name": "since", "type": "INTEGER"}
        ]
    },
    "USES"
]

print("✓ 상세 스키마 정의 완료")

✓ 상세 스키마 정의 완료


### **4.3 GraphSchema 클래스 사용 (고급)**

가장 엄격한 스키마 정의 방법으로, 타입 체계를 명확하게 정의합니다.

In [10]:
from neo4j_graphrag.experimental.components.schema import (
    GraphSchema, NodeType, RelationshipType, PropertyType
)

# GraphSchema 객체로 정의
schema = GraphSchema(
    node_types=[
        NodeType(
            label="Person",
            properties=[
                PropertyType(name="name", type="STRING"),
                PropertyType(name="role", type="STRING"),
            ]
        ),
        NodeType(label="Company"),
        NodeType(label="Technology"),
        NodeType(label="Product"),
    ],
    relationship_types=[
        RelationshipType(
            label="WORKS_FOR",
            source="Person",
            target="Company",
        ),
        RelationshipType(
            label="DEVELOPS",
            source="Company",
            target="Technology",
        ),
    ]
)

print("✓ GraphSchema 객체 생성 완료")
print("\n💡 참고: GraphSchema를 사용할 때는 schema 파라미터만 전달하세요.")
print("   entities, relations, potential_schema 파라미터와 함께 사용하면 안 됩니다.")

✓ GraphSchema 객체 생성 완료

💡 참고: GraphSchema를 사용할 때는 schema 파라미터만 전달하세요.
   entities, relations, potential_schema 파라미터와 함께 사용하면 안 됩니다.


---

## **5. SimpleKGPipeline 구성 및 실행**

### **5.1 기본 파이프라인 생성**

In [11]:
# SimpleKGPipeline 초기화
kg_builder = SimpleKGPipeline(
    # 필수 파라미터
    llm=llm,
    driver=driver,
    embedder=embedder,
    
    # 스키마 설정 (기본 문자열 방식)
    entities=entities,
    relations=relations,
    potential_schema=potential_schema,
    
    # 텍스트 분할 설정
    text_splitter=FixedSizeSplitter(
        chunk_size=500,      # 청크 크기 (문자 수)
        chunk_overlap=100    # 청크 간 겹침 (문자 수)
    ),
    
    # 입력 소스 설정 (중요)
    # - from_file=False: text= 로 원문을 직접 전달할 때 사용 (본 예시)
    # - from_file=True : file_path= 로 PDF/Markdown 파일을 전달할 때 사용
    # neo4j-graphrag 1.15+ 에서 기본값이 True 로 변경되었으므로 텍스트 입력 시 명시 필요
    from_file=False,
    
    # Entity Resolution 활성화 (중복 엔티티 병합)
    perform_entity_resolution=True,
    
    # 에러 처리 방식
    # - "IGNORE": 추출 실패한 청크는 건너뛰고 계속 진행 (권장)
    # - "RAISE": 에러 발생 시 즉시 예외를 발생시키고 중단
    on_error="IGNORE",
    
    # Neo4j 데이터베이스 지정
    neo4j_database=NEO4J_DATABASE
)

print("✓ SimpleKGPipeline 생성 완료")

✓ SimpleKGPipeline 생성 완료


### **5.2 샘플 뉴스 기사 준비**

In [12]:
# 실습용 뉴스 기사 텍스트
news_text = """
엔비디아, AI 반도체 시장 선도

엔비디아(NVIDIA)가 인공지능(AI) 반도체 시장에서 압도적인 점유율을 기록하고 있다. 
젠슨 황(Jensen Huang) CEO는 최근 실적 발표회에서 "AI 수요가 폭발적으로 증가하고 있으며, 
우리의 GPU 기술은 이러한 수요를 충족시키는 핵심"이라고 밝혔다.

엔비디아의 대표 제품인 H100 GPU는 대규모 언어 모델(LLM) 학습에 최적화되어 있으며, 
OpenAI, 구글, 메타 등 주요 AI 기업들이 H100을 대량으로 도입하고 있다. 
업계 전문가들은 엔비디아의 시장 점유율이 80%를 넘어설 것으로 전망하고 있다.

한편, AMD는 MI300 시리즈를 출시하며 엔비디아에 도전장을 내밀었다. 
AMD의 리사 수(Lisa Su) CEO는 "MI300은 H100과 직접 경쟁할 수 있는 성능을 제공한다"며 
시장 점유율 확대를 목표로 하고 있다고 밝혔다.

삼성전자와 SK하이닉스도 HBM(고대역폭 메모리) 시장에서 중요한 역할을 하고 있다. 
SK하이닉스는 엔비디아 H100에 탑재되는 HBM3를 독점 공급하며 큰 수익을 올리고 있으며, 
삼성전자는 차세대 HBM3E 개발에 박차를 가하고 있다.
"""

print("✓ 샘플 텍스트 준비 완료")
print(f"  텍스트 길이: {len(news_text)} 문자")

✓ 샘플 텍스트 준비 완료
  텍스트 길이: 590 문자


### **5.3 Knowledge Graph 자동 생성**

In [13]:
# 비동기 실행 (async)
import asyncio

async def build_kg():
    """Knowledge Graph 생성 함수"""
    result = await kg_builder.run_async(
        text=news_text
    )
    return result

# 실행
result = await build_kg()
print("\n✓ Knowledge Graph 생성 완료!")
print(f"  처리 결과: {result}")


✓ Knowledge Graph 생성 완료!
  처리 결과: run_id='b036697d-eca0-43f7-8b3f-c3fe09460031' result={'resolver': {'number_of_nodes_to_resolve': 18, 'number_of_created_nodes': 15}}


### **5.4 두 번째 기사 추가 (Entity Resolution 연습용)**

같은 회사·인물·기술을 **영문/변형 표기**로 쓴 두 번째 기사를 한 번 더 파이프라인에 태웁니다.

| 표기 유형 | 첫 기사 | 두 번째 기사 |
|---|---|---|
| 회사 | 엔비디아 · AMD · 삼성전자 | **NVIDIA** · **Nvidia Corporation** · **Samsung Electronics** |
| 인물 | 젠슨 황 · 리사 수 | **Jensen Huang** · **Lisa Su** |
| 기술/제품 | H100 GPU · HBM3 | **H100** · **HBM** |

<br>

> 🔑 **이 단계의 목적**
> - `perform_entity_resolution=True` 는 **이름이 정확히 같을 때만** 병합합니다 (built-in `SinglePropertyExactMatchResolver`).
> - 따라서 `엔비디아` 와 `NVIDIA` 는 **서로 다른 노드** 로 남습니다. 이걸 다음 7.1 섹션에서 직접 확인하고, 7.2 에서 해결합니다.

In [14]:
# 두 번째 기사 — 동일 주제를 영문/변형 표기로 서술
news_text_en = """
NVIDIA Dominates AI Chip Market

NVIDIA Corporation continues to lead the AI chip market, with CEO Jensen Huang recently announcing record-breaking H100 sales. 
"AI demand is exploding, and our GPU technology is the backbone of that growth," said Jensen Huang at the earnings call.

Major customers such as OpenAI, Google, and Meta have deployed tens of thousands of H100 units. 
Analysts expect Nvidia's market share to exceed 80 percent this year.

Meanwhile, AMD's MI300 series is positioning itself as the main competitor. AMD CEO Lisa Su stated,
"MI300 can go head-to-head with H100 in both performance and total cost of ownership."

On the memory side, Samsung Electronics and SK Hynix remain critical HBM suppliers. 
SK Hynix has an exclusive supply of HBM3 for NVIDIA's H100, 
and Samsung Electronics is accelerating development of next-generation HBM3E.
"""

print(f"✓ 두 번째 기사 준비 완료 — {len(news_text_en)} 문자")

# 기존 kg_builder 재활용 (동일한 Neo4j DB 에 누적 적재)
async def build_kg_en():
    result = await kg_builder.run_async(text=news_text_en)
    return result

result_en = await build_kg_en()
print("\n✓ 두 번째 Knowledge Graph 적재 완료")
print(f"  처리 결과: {result_en}")

✓ 두 번째 기사 준비 완료 — 864 문자

✓ 두 번째 Knowledge Graph 적재 완료
  처리 결과: run_id='c1f913de-67ff-42f7-ad50-3db51f629ccd' result={'resolver': {'number_of_nodes_to_resolve': 31, 'number_of_created_nodes': 24}}


---

## **6. 생성된 Knowledge Graph 검증**

### **6.1 스키마 확인**

In [15]:
# 그래프 스키마 갱신 및 출력
graph.refresh_schema()
print("생성된 그래프 스키마:")
print(graph.schema)

생성된 그래프 스키마:
Node properties:
Person {name: STRING}
Company {name: STRING}
Product {name: STRING}
Technology {name: STRING}
Document {path: STRING, createdAt: STRING, document_type: STRING}
Chunk {index: INTEGER, text: STRING, embedding: LIST}
Relationship properties:

The relationships:
(:Person)-[:WORKS_FOR]->(:Company)
(:Person)-[:FROM_CHUNK]->(:Chunk)
(:Company)-[:COMPETES_WITH]->(:Company)
(:Company)-[:FROM_CHUNK]->(:Chunk)
(:Company)-[:DEVELOPS]->(:Product)
(:Company)-[:DEVELOPS]->(:Technology)
(:Product)-[:FROM_CHUNK]->(:Chunk)
(:Product)-[:USES]->(:Technology)
(:Technology)-[:FROM_CHUNK]->(:Chunk)
(:Chunk)-[:FROM_DOCUMENT]->(:Document)
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)


### **6.2 노드 통계 확인**

In [16]:
# 노드 타입별 개수 확인
node_count_query = """
MATCH (n)
RETURN labels(n)[0] as nodeType, count(n) as count
ORDER BY count DESC
"""

node_counts = graph.query(node_count_query)
print("\n노드 타입별 통계:")
for record in node_counts:
    print(f"  {record['nodeType']}: {record['count']}개")


노드 타입별 통계:
  Company: 13개
  __KGBuilder__: 6개
  Person: 4개
  Product: 4개
  Technology: 3개


### **6.3 관계 통계 확인**

In [17]:
# 관계 타입별 개수 확인
rel_count_query = """
MATCH ()-[r]->()
RETURN type(r) as relationType, count(r) as count
ORDER BY count DESC
"""

rel_counts = graph.query(rel_count_query)
print("\n관계 타입별 통계:")
for record in rel_counts:
    print(f"  {record['relationType']}: {record['count']}개")


관계 타입별 통계:
  FROM_CHUNK: 34개
  DEVELOPS: 13개
  FROM_DOCUMENT: 4개
  WORKS_FOR: 4개
  NEXT_CHUNK: 2개
  COMPETES_WITH: 2개
  USES: 1개


### **6.4 추출된 엔티티 확인**

In [18]:
# Company 엔티티 확인
companies_query = """
MATCH (c:Company)
RETURN c.name as company
ORDER BY company
"""

companies = graph.query(companies_query)
print("\n추출된 회사 엔티티:")
for record in companies:
    print(f"  - {record['company']}")


추출된 회사 엔티티:
  - AMD
  - Google
  - Meta
  - NVIDIA Corporation
  - Nvidia
  - OpenAI
  - SK Hynix
  - SK하이닉스
  - Samsung Electronics
  - 구글
  - 메타
  - 삼성전자
  - 엔비디아


In [19]:
# Person 엔티티 확인
persons_query = """
MATCH (p:Person)
RETURN p.name as person, p.role as role
ORDER BY person
"""

persons = graph.query(persons_query)
print("\n추출된 인물 엔티티:")
for record in persons:
    role = record.get('role', '알 수 없음')
    print(f"  - {record['person']} ({role})")


추출된 인물 엔티티:
  - Jensen Huang (None)
  - Lisa Su (None)
  - 리사 수 (None)
  - 젠슨 황 (None)


In [20]:
# Technology 엔티티 확인
tech_query = """
MATCH (t:Technology)
RETURN t.name as technology
ORDER BY technology
"""

technologies = graph.query(tech_query)
print("\n추출된 기술 엔티티:")
for record in technologies:
    print(f"  - {record['technology']}")


추출된 기술 엔티티:
  - HBM
  - HBM3
  - HBM3E


### **6.5 관계 네트워크 확인**

In [21]:
# 회사 간 경쟁 관계 확인
competition_query = """
MATCH (c1:Company)-[r:COMPETES_WITH]->(c2:Company)
RETURN c1.name as company1, c2.name as company2
"""

competitions = graph.query(competition_query)
print("\n회사 간 경쟁 관계:")
for record in competitions:
    print(f"  {record['company1']} ⟷ {record['company2']}")


회사 간 경쟁 관계:
  엔비디아 ⟷ AMD
  AMD ⟷ Nvidia


In [22]:
# 회사와 기술 개발 관계 확인
development_query = """
MATCH (c:Company)-[r:DEVELOPS]->(t)
RETURN c.name as company, labels(t)[0] as type, t.name as technology
ORDER BY company
"""

developments = graph.query(development_query)
print("\n회사의 기술/제품 개발 관계:")
for record in developments:
    print(f"  {record['company']} → {record['technology']} ({record['type']})")


회사의 기술/제품 개발 관계:
  AMD → MI300 시리즈 (Product)
  AMD → MI300 (Product)
  NVIDIA Corporation → H100 (Product)
  Nvidia → H100 (Product)
  SK Hynix → HBM (Technology)
  SK Hynix → HBM3 (Technology)
  SK하이닉스 → HBM (Technology)
  SK하이닉스 → HBM3 (Technology)
  Samsung Electronics → HBM (Technology)
  Samsung Electronics → HBM3E (Technology)
  삼성전자 → HBM (Technology)
  삼성전자 → HBM3E (Technology)
  엔비디아 → H100 GPU (Product)


### **6.6 Lexical Graph 구조 확인**

In [23]:
# Document → Chunk → Entity 경로 확인
# ⚠️ 관계 방향: Chunk-[:FROM_DOCUMENT]->Document, Entity-[:FROM_CHUNK]->Chunk
lexical_query = """
MATCH (chunk:Chunk)-[:FROM_DOCUMENT]->(d:Document)
OPTIONAL MATCH (entity)-[:FROM_CHUNK]->(chunk)
WHERE entity:Company OR entity:Person OR entity:Technology
RETURN
    chunk.index AS chunkIndex,
    substring(chunk.text, 0, 100) + '...' AS chunkPreview,
    collect(DISTINCT labels(entity)[0] + ': ' + entity.name) AS entities
ORDER BY chunkIndex
LIMIT 5
"""

lexical_graph = graph.query(lexical_query)
print("\nLexical Graph 구조 (청크별 추출 엔티티):")
for record in lexical_graph:
    ents = [e for e in record['entities'] if e and not e.startswith('None')]
    print(f"\nChunk {record['chunkIndex']}:")
    print(f"  텍스트: {record['chunkPreview']}")
    print(f"  추출된 엔티티: {', '.join(ents) if ents else '없음'}")


Lexical Graph 구조 (청크별 추출 엔티티):

Chunk 0:
  텍스트: 
엔비디아, AI 반도체 시장 선도

엔비디아(NVIDIA)가 인공지능(AI) 반도체 시장에서 압도적인 점유율을 기록하고 있다. 
젠슨 황(Jensen Huang) CEO는 최근 ...
  추출된 엔티티: Technology: HBM, Company: 엔비디아, Person: 젠슨 황, Company: OpenAI, Company: 구글, Company: 메타, Company: AMD, Person: 리사 수, Company: 삼성전자, Company: SK하이닉스

Chunk 0:
  텍스트: 
NVIDIA Dominates AI Chip Market

NVIDIA Corporation continues to lead the AI chip market, with CEO ...
  추출된 엔티티: Company: OpenAI, Company: Meta, Company: NVIDIA Corporation, Person: Jensen Huang, Company: Google

Chunk 1:
  텍스트: H100과 직접 경쟁할 수 있는 성능을 제공한다"며 
시장 점유율 확대를 목표로 하고 있다고 밝혔다.

삼성전자와 SK하이닉스도 HBM(고대역폭 메모리) 시장에서 중요한 역할을 하...
  추출된 엔티티: Technology: HBM, Technology: HBM3, Technology: HBM3E, Company: 삼성전자, Company: SK하이닉스

Chunk 1:
  텍스트: Nvidia's market share to exceed 80 percent this year.

Meanwhile, AMD's MI300 series is positioning ...
  추출된 엔티티: Technology: HBM, Technology: HBM3, Technology: HBM3E, Company: AMD, Person: Lisa Su, Company: Samsung Electr

---

## **7. Entity Resolution 실습**

### **7.1 Entity Resolution이란?**

동일한 엔티티가 텍스트의 여러 부분에서 다르게 표현될 때 (예: "엔비디아", "NVIDIA", "Nvidia Corporation"), 이를 하나의 노드로 병합하는 기능입니다.

#### SimpleKGPipeline 의 기본 Resolution (`SinglePropertyExactMatchResolver`)
- **이름 기반 정확 일치**: `label` + `name` 이 **완전히 같을 때만** 병합
- `perform_entity_resolution=True` 로 자동 실행 (본 노트북 기본값)
- ⚠️ `엔비디아` vs `NVIDIA` 처럼 **표기가 다르면** 기본 Resolver 로는 병합되지 않습니다 → 7.2 의 Fuzzy / 수동 병합 필요

> 🔑 **Entity Resolution 없으면 벌어지는 일**
>
> 문서가 A·B 두 편이 있고 각각 "엔비디아"·"NVIDIA" 로 표기됐다면, 
> 기본 추출만으로는 **별도의 두 Company 노드** 가 생깁니다. 
> 이후 "엔비디아가 개발한 기술" 쿼리는 B 문서 내용을 놓치게 됩니다.
>
> ⚠️ `perform_entity_resolution=True` 를 켜면 SimpleKGPipeline 이 
> "같은 라벨 + 이름 유사도" 기준으로 **자동 병합** 해줍니다. 
> 한글·영문 혼용처럼 복잡한 경우는 **Fuzzy Resolver** 또는 **별도 후처리 Cypher** 가 필요합니다.


In [24]:
# [1] 정확 이름 중복 — 기본 Resolver 가 이미 병합했으므로 보통 0건
exact_dup_query = """
MATCH (n)
WHERE n:Company OR n:Person OR n:Technology OR n:Product
WITH labels(n)[0] AS type, n.name AS name, count(*) AS duplicates
WHERE duplicates > 1
RETURN type, name, duplicates
ORDER BY duplicates DESC
"""

print("=" * 60)
print("[1] 정확 이름 중복 (built-in resolver 작동 확인)")
print("=" * 60)
exact_dups = graph.query(exact_dup_query)
if exact_dups:
    for r in exact_dups:
        print(f"  {r['type']}: {r['name']} ({r['duplicates']}개)")
else:
    print("✓ 정확 이름 중복 0건 — 내장 Resolver 가 같은 이름은 모두 병합")

# [2] 유사 이름 후보 — 변형 표기(엔비디아/NVIDIA, Samsung Electronics/삼성전자 등) 찾기
# 같은 라벨 안에서 소문자·공백 제거 기준으로 포함 관계인 노드 쌍을 묶어줍니다
similar_name_query = """
MATCH (a), (b)
WHERE (a:Company OR a:Person OR a:Technology OR a:Product)
  AND labels(a) = labels(b)
  AND elementId(a) < elementId(b)
  AND a.name IS NOT NULL AND b.name IS NOT NULL
WITH labels(a)[0] AS type, a.name AS nameA, b.name AS nameB,
     toLower(replace(a.name, ' ', '')) AS keyA,
     toLower(replace(b.name, ' ', '')) AS keyB
WHERE keyA CONTAINS keyB OR keyB CONTAINS keyA
RETURN type, nameA, nameB
ORDER BY type, nameA
"""

print("\n" + "=" * 60)
print("[2] 유사 이름 후보 (FuzzyMatchResolver 또는 수동 병합 대상)")
print("=" * 60)
similar = graph.query(similar_name_query)
if similar:
    for r in similar:
        print(f"  [{r['type']}] '{r['nameA']}'  ↔  '{r['nameB']}'")
else:
    print("  유사 이름 후보 없음")

[1] 정확 이름 중복 (built-in resolver 작동 확인)
✓ 정확 이름 중복 0건 — 내장 Resolver 가 같은 이름은 모두 병합

[2] 유사 이름 후보 (FuzzyMatchResolver 또는 수동 병합 대상)
  [Company] 'NVIDIA Corporation'  ↔  'Nvidia'
  [Product] 'H100 GPU'  ↔  'H100'
  [Product] 'MI300 시리즈'  ↔  'MI300'
  [Technology] 'HBM'  ↔  'HBM3'
  [Technology] 'HBM'  ↔  'HBM3E'
  [Technology] 'HBM3'  ↔  'HBM3E'


### **7.2 Fuzzy Matching Resolver (RapidFuzz 기반)**

`neo4j-graphrag` 가 제공하는 `FuzzyMatchResolver` 는 **RapidFuzz** 의 `fuzz.WRatio` 를 이용해 같은 라벨 내에서 이름이 유사한 노드를 자동 병합합니다.

- `similarity_threshold=0.8` : 0~1 사이 점수가 임계치 이상이면 병합
- 내부적으로 **소문자 변환 + 구두점 제거 + 공백 정리** (`utils.default_process`) 전처리를 거친 후 비교
- 장점: 설치 한 번으로 끝, 프롬프트 비용 0원  
- 한계: 문자 기반이라 **한글 ↔ 영문** 변환(`엔비디아` ↔ `NVIDIA`) 같은 **의미적 매칭은 불가**

In [25]:
# 드라이버가 닫혀 있으면 자동 재연결
ensure_neo4j_connection()

# 7.2 Fuzzy Matching Resolver 실행
# 1) 필요 패키지 설치 (최초 1회만 실행)
#    pip install "neo4j-graphrag[fuzzy-matching]"
# Colab/Jupyter 에서 바로 설치하려면 아래 주석 해제
# %pip install -q "neo4j-graphrag[fuzzy-matching]"

from neo4j_graphrag.experimental.components.resolver import FuzzyMatchResolver

# 공통: 라벨별 엔티티 이름 스냅샷 유틸 (이후 섹션에서도 재사용)
SNAPSHOT_LABELS = ["Company", "Person", "Technology", "Product"]

def snapshot_entities():
    snap = {}
    for label in SNAPSHOT_LABELS:
        rows = graph.query(
            f"MATCH (n:`{label}`) WHERE n.name IS NOT NULL "
            "RETURN n.name AS name ORDER BY name"
        )
        snap[label] = sorted({r["name"] for r in rows})
    return snap

# [1] 실행 전 스냅샷
before = snapshot_entities()
print("=" * 60)
print("Fuzzy 실행 전 엔티티 스냅샷")
print("=" * 60)
for label, names in before.items():
    print(f"  [{label}] ({len(names)}개): {names}")

# [2] Fuzzy 실행
fuzzy_resolver = FuzzyMatchResolver(
    driver=driver,
    resolve_properties=["name"],
    similarity_threshold=0.80,
    neo4j_database=NEO4J_DATABASE,
)
stats = await fuzzy_resolver.run()
print(f"\n✓ Fuzzy 실행 완료 — 검사 {stats.number_of_nodes_to_resolve}개 → 남은 {stats.number_of_created_nodes}개")

# [3] 실행 후 스냅샷 + Before/After diff
after = snapshot_entities()
print("\n" + "=" * 60)
print("병합 결과 (라벨별 Before → After)")
print("=" * 60)
any_change = False
for label in SNAPSHOT_LABELS:
    removed = sorted(set(before[label]) - set(after[label]))
    added   = sorted(set(after[label]) - set(before[label]))   # 보통 비어 있음
    if removed or added:
        any_change = True
        print(f"\n[{label}]")
        print(f"  Before ({len(before[label])}개): {before[label]}")
        print(f"  After  ({len(after[label])}개): {after[label]}")
        if removed:
            print(f"  → 병합되어 사라진 이름: {removed}")
        if added:
            print(f"  → 새로 생긴 이름: {added}")
if not any_change:
    print("  변화 없음 (병합된 노드 없음)")

# [4] 유사 이름 후보 재점검
print("\n" + "=" * 60)
print("Fuzzy 병합 후 남은 유사 이름 후보")
print("=" * 60)
after_pairs = graph.query(similar_name_query)
if after_pairs:
    for r in after_pairs:
        print(f"  [{r['type']}] '{r['nameA']}'  ↔  '{r['nameB']}'")
else:
    print("  ✓ 남은 유사 이름 후보 없음 (한글↔영문 매칭은 7.3 LLM 으로 처리)")

✓ Neo4j 연결 상태 정상
Fuzzy 실행 전 엔티티 스냅샷
  [Company] (13개): ['AMD', 'Google', 'Meta', 'NVIDIA Corporation', 'Nvidia', 'OpenAI', 'SK Hynix', 'SK하이닉스', 'Samsung Electronics', '구글', '메타', '삼성전자', '엔비디아']
  [Person] (4개): ['Jensen Huang', 'Lisa Su', '리사 수', '젠슨 황']
  [Technology] (3개): ['HBM', 'HBM3', 'HBM3E']
  [Product] (4개): ['H100', 'H100 GPU', 'MI300', 'MI300 시리즈']

✓ Fuzzy 실행 완료 — 검사 24개 → 남은 4개

병합 결과 (라벨별 Before → After)

[Company]
  Before (13개): ['AMD', 'Google', 'Meta', 'NVIDIA Corporation', 'Nvidia', 'OpenAI', 'SK Hynix', 'SK하이닉스', 'Samsung Electronics', '구글', '메타', '삼성전자', '엔비디아']
  After  (12개): ['AMD', 'Google', 'Meta', 'NVIDIA Corporation', 'OpenAI', 'SK Hynix', 'SK하이닉스', 'Samsung Electronics', '구글', '메타', '삼성전자', '엔비디아']
  → 병합되어 사라진 이름: ['Nvidia']

[Technology]
  Before (3개): ['HBM', 'HBM3', 'HBM3E']
  After  (1개): ['HBM']
  → 병합되어 사라진 이름: ['HBM3', 'HBM3E']

[Product]
  Before (4개): ['H100', 'H100 GPU', 'MI300', 'MI300 시리즈']
  After  (2개): ['H100 GPU', 'MI300 시리즈']
  → 병합되어 사라

### **7.3 LLM 기반 Entity Resolution**

RapidFuzz 로도 잡히지 않는 **한글↔영문**, **약어↔정식명** 같은 **의미적 동일성** 은 LLM 에게 물어보는 게 가장 실용적입니다.

**처리 흐름**
1. 라벨별로 DB 에 존재하는 이름 목록을 수집
2. LLM 에게 "어떤 이름들이 같은 엔티티인지" JSON 으로 묶게 요청
3. 각 그룹에서 canonical 을 선택하고 나머지를 `apoc.refactor.mergeNodes` 로 병합

**장·단점**
- ✅ 언어·표기 차이까지 커버 (`엔비디아` ↔ `NVIDIA Corporation`)
- ⚠️ LLM 호출 비용 + 환각 가능성 → 프로덕션에서는 병합 전 사람 승인 워크플로 권장

In [26]:
# 드라이버가 닫혀 있으면 자동 재연결
ensure_neo4j_connection()

# 7.3 LLM 기반 Entity Resolution 실행
import json

# [0] 실행 전 스냅샷 (7.2 의 snapshot_entities 재사용)
before_llm = snapshot_entities()
print("=" * 60)
print("LLM 병합 실행 전 엔티티 스냅샷")
print("=" * 60)
for label, names in before_llm.items():
    print(f"  [{label}] ({len(names)}개): {names}")

# [1] 라벨별로 현재 DB 의 이름 목록 수집
target_labels = ["Company", "Person", "Technology", "Product"]
label_to_names = {}
for label in target_labels:
    rows = graph.query(f"MATCH (n:`{label}`) WHERE n.name IS NOT NULL RETURN collect(DISTINCT n.name) AS names")
    names = rows[0]["names"] if rows else []
    if len(names) >= 2:
        label_to_names[label] = names

if not label_to_names:
    print("\n⚠️ 병합 후보가 남아 있지 않아 LLM 단계를 건너뜁니다.")
    resolution_plan = {}
else:
    # [2] LLM 에게 동일 엔티티 그룹을 JSON 으로 받기
    resolution_prompt = f"""당신은 Knowledge Graph 의 Entity Resolution 을 돕는 도구입니다.
아래 라벨별 이름 목록을 보고, **같은 실체(entity)를 가리키는 이름들을 하나의 그룹으로 묶어 주세요**.

🚨 절대 규칙:
1. `canonical` 과 `aliases` 안의 **모든 이름은 반드시 아래 입력 목록에 실제로 존재하는 이름** 이어야 합니다. 
   새 이름을 만들어 내지 마세요. 예: 입력에 "NVIDIA" 만 있고 "엔비디아" 가 없다면 canonical 은 "NVIDIA" 입니다.
2. 한글·영문·약어·대소문자 차이만 있는 경우는 같은 그룹으로 묶으세요 (예: 입력에 "엔비디아" 와 "NVIDIA" 가 함께 있으면 같은 그룹).
3. **다른 제품/세대는 묶지 마세요** (예: "HBM3" != "HBM3E", "H100" != "H200").
4. 각 그룹에서 canonical 을 선택할 때는: (a) 한글 정식 회사명/인명이 입력에 있으면 그것을, (b) 없으면 가장 **정식에 가까운 영문 표기** 를 선택하세요.
5. 이름이 하나뿐이거나 묶을 쌍이 없으면 그 라벨은 결과에서 생략하세요.

입력 데이터:
{json.dumps(label_to_names, ensure_ascii=False, indent=2)}

출력 형식 (JSON 만, 설명 없이):
{{
  "Company": [
    {{"canonical": "<입력에 존재하는 이름>", "aliases": ["<입력에 존재하는 다른 이름>", ...]}}
  ],
  "Person": [ ... ]
}}
"""

    response = await llm.ainvoke(resolution_prompt)
    print("\n" + "=" * 60)
    print("LLM 응답")
    print("=" * 60)
    print(response.content)

    # [3] LLM 응답 파싱 + 검증 (입력 목록에 없는 이름은 자동 치환/제거)
    raw_plan = json.loads(response.content)
    resolution_plan = {}
    dropped_warnings = []
    for label, groups in raw_plan.items():
        existing = set(label_to_names.get(label, []))
        cleaned_groups = []
        for g in groups:
            canonical = g.get("canonical", "")
            aliases = [a for a in g.get("aliases", []) if a in existing and a != canonical]
            if canonical not in existing:
                # canonical 이 DB 에 없으면 aliases 중 첫 번째를 canonical 로 승격
                if aliases:
                    dropped_warnings.append(f"[{label}] canonical '{canonical}' 이 DB 에 없어 '{aliases[0]}' 로 교체")
                    canonical = aliases[0]
                    aliases = aliases[1:]
                else:
                    dropped_warnings.append(f"[{label}] canonical '{canonical}' 과 aliases 모두 DB 에 없어 그룹 생략")
                    continue
            if aliases:  # 병합할 alias 가 있을 때만 계획에 포함
                cleaned_groups.append({"canonical": canonical, "aliases": aliases})
        if cleaned_groups:
            resolution_plan[label] = cleaned_groups

    if dropped_warnings:
        print("\n⚠️ LLM 응답 보정:")
        for w in dropped_warnings:
            print(f"    {w}")

    print("\n정제된 병합 계획:")
    print(json.dumps(resolution_plan, ensure_ascii=False, indent=2))

    # [4] APOC 로 병합 수행 (elementId 사용으로 deprecation 경고 제거)
    apoc_merge_q = """
    MATCH (c) WHERE $label IN labels(c) AND c.name = $canonical
    MATCH (a) WHERE $label IN labels(a) AND a.name = $alias
    WITH c, a WHERE elementId(c) <> elementId(a)
    CALL apoc.refactor.mergeNodes([c, a], {properties: 'discard', mergeRels: true}) YIELD node
    RETURN node.name AS name
    """

    print("\n" + "=" * 60)
    print("LLM 결정에 따른 병합 실행")
    print("=" * 60)
    merged_total = 0
    for label, groups in resolution_plan.items():
        for group in groups:
            canonical = group["canonical"]
            for alias in group["aliases"]:
                res = graph.query(apoc_merge_q, {"label": label, "canonical": canonical, "alias": alias})
                if res:
                    merged_total += 1
                    print(f"  ✓ [{label}] '{alias}' → '{canonical}'")
                else:
                    print(f"  ⚠️ [{label}] '{alias}' → '{canonical}' 병합 실패 (노드를 찾지 못함)")

    print(f"\n총 {merged_total}건 LLM 기반 병합 완료")

# [5] 실행 후 스냅샷 + Before/After diff
after_llm = snapshot_entities()
print("\n" + "=" * 60)
print("LLM 병합 결과 (라벨별 Before → After)")
print("=" * 60)
any_change = False
for label in SNAPSHOT_LABELS:
    removed = sorted(set(before_llm[label]) - set(after_llm[label]))
    if removed:
        any_change = True
        print(f"\n[{label}]")
        print(f"  Before ({len(before_llm[label])}개): {before_llm[label]}")
        print(f"  After  ({len(after_llm[label])}개): {after_llm[label]}")
        print(f"  → 병합되어 사라진 이름: {removed}")
if not any_change:
    print("  변화 없음 (추가 병합된 노드 없음)")

# [6] 최종 재점검
print("\n" + "=" * 60)
print("LLM 병합 후 남은 유사 이름 후보")
print("=" * 60)
after_pairs = graph.query(similar_name_query)
if after_pairs:
    for r in after_pairs:
        print(f"  [{r['type']}] '{r['nameA']}'  ↔  '{r['nameB']}'")
else:
    print("  ✓ 남은 유사 이름 후보 없음")

✓ Neo4j 연결 상태 정상
LLM 병합 실행 전 엔티티 스냅샷
  [Company] (12개): ['AMD', 'Google', 'Meta', 'NVIDIA Corporation', 'OpenAI', 'SK Hynix', 'SK하이닉스', 'Samsung Electronics', '구글', '메타', '삼성전자', '엔비디아']
  [Person] (4개): ['Jensen Huang', 'Lisa Su', '리사 수', '젠슨 황']
  [Technology] (1개): ['HBM']
  [Product] (2개): ['H100 GPU', 'MI300 시리즈']

LLM 응답
{
  "Company": [
    {"canonical": "엔비디아", "aliases": ["NVIDIA Corporation"]},
    {"canonical": "구글", "aliases": ["Google"]},
    {"canonical": "메타", "aliases": ["Meta"]},
    {"canonical": "삼성전자", "aliases": ["Samsung Electronics"]},
    {"canonical": "SK하이닉스", "aliases": ["SK Hynix"]}
  ],
  "Person": [
    {"canonical": "젠슨 황", "aliases": ["Jensen Huang"]},
    {"canonical": "리사 수", "aliases": ["Lisa Su"]}
  ]
}

정제된 병합 계획:
{
  "Company": [
    {
      "canonical": "엔비디아",
      "aliases": [
        "NVIDIA Corporation"
      ]
    },
    {
      "canonical": "구글",
      "aliases": [
        "Google"
      ]
    },
    {
      "canonical": "메타",
      "aliase

### **7.4 수동 Cypher 병합 (폴백)**

Fuzzy / LLM 이 모두 어려운 환경(오프라인, 예산 제약)에서는 **별칭 대응표를 직접 코드에 정의** 해 APOC 로 병합합니다. 본 실습 그래프처럼 범위가 좁을 때 가장 확실한 방법.

In [ ]:
# 드라이버가 닫혀 있으면 자동 재연결
ensure_neo4j_connection()

# 수동 Cypher 병합 — APOC 의 apoc.refactor.mergeNodes 사용
# AuraDB Free 는 APOC Core 가 기본 포함되어 별도 설치 불필요
# (Fuzzy 라이브러리 설치가 부담스러운 교실 환경을 위한 대안)

# 한글·영문 별칭 대응표 (실무에서는 LLM 또는 FuzzyMatchResolver 로 자동화)
alias_map = [
    # (label, canonical, [aliases ...])
    ("Company",    "엔비디아",   ["NVIDIA", "Nvidia Corporation", "Nvidia"]),
    ("Company",    "삼성전자",   ["Samsung Electronics", "Samsung"]),
    ("Company",    "SK하이닉스", ["SK Hynix"]),
    ("Person",     "젠슨 황",    ["Jensen Huang"]),
    ("Person",     "리사 수",    ["Lisa Su"]),
    ("Technology", "H100",      ["H100 GPU"]),
]

merge_query = """
MATCH (c) WHERE $canonical_label IN labels(c) AND c.name = $canonical
MATCH (a) WHERE $alias_label     IN labels(a) AND a.name = $alias
WITH c, a WHERE elementId(c) <> elementId(a)
CALL apoc.refactor.mergeNodes([c, a], {
    properties: 'discard',
    mergeRels: true
}) YIELD node
RETURN node.name AS name
"""

merged_count = 0
for label, canonical, aliases in alias_map:
    for alias in aliases:
        res = graph.query(merge_query, {
            "canonical_label": label,
            "alias_label": label,
            "canonical": canonical,
            "alias": alias,
        })
        if res:
            merged_count += 1
            print(f"  ✓ [{label}] '{alias}' → '{canonical}' 병합")

print(f"\n총 {merged_count}건 병합 완료")

# 병합 결과 재점검
print("\n" + "=" * 60)
print("병합 후 유사 이름 후보 재점검")
print("=" * 60)
after = graph.query(similar_name_query)
if after:
    for r in after:
        print(f"  [{r['type']}] '{r['nameA']}'  ↔  '{r['nameB']}'")
else:
    print("  ✓ 남은 유사 이름 후보 없음")

---

## **8. 실습 문제**

### **실습 1: 다른 뉴스 기사로 Knowledge Graph 확장**

아래 뉴스 기사를 사용하여 기존 Knowledge Graph에 새로운 정보를 추가하세요.

In [ ]:
# 실습용 추가 뉴스 기사
additional_news = """
OpenAI, GPT-5 개발 착수

샘 알트먼(Sam Altman) OpenAI CEO는 차세대 언어 모델 GPT-5 개발에 착수했다고 밝혔다. 
GPT-5는 GPT-4보다 10배 이상의 파라미터를 가질 것으로 예상되며, 
이를 위해 엔비디아로부터 수만 대의 H100 GPU를 도입할 계획이다.

마이크로소프트는 OpenAI의 주요 투자자이자 파트너로서 Azure 클라우드를 통해 
GPT-5 학습을 위한 인프라를 제공할 예정이다. 업계에서는 마이크로소프트가 
추가로 100억 달러를 투자할 것으로 관측하고 있다.

한편 구글의 딥마인드(DeepMind)는 Gemini Pro의 후속 모델 개발을 진행 중이며, 
데미스 하사비스(Demis Hassabis) CEO는 "Gemini Ultra 2가 GPT-5와 경쟁할 것"이라고 언급했다.
"""

# 여기에 코드를 작성하세요.
# kg_builder.run_async()를 사용하여 additional_news를 처리하세요.

async def extend_kg():
    # TODO: 파이프라인 실행 코드 작성
    pass

# result = await extend_kg()
# print("실습 1 완료:", result)


<details close>
<summary>💡 정답 확인</summary>

```python
기존 파이프라인을 사용하여 추가 뉴스 처리
async def extend_kg():
    """기존 Knowledge Graph에 새로운 뉴스 정보 추가"""
    result = await kg_builder.run_async(
        text=additional_news
    )
    return result

실행
result = await extend_kg()
print("✓ 실습 1 완료: Knowledge Graph 확장 성공!")
print(f"  처리 결과: {result}")

확장된 그래프 확인
print("\n추가된 회사 엔티티 확인:")
new_companies_query = """
MATCH (c:Company)
WHERE c.name IN ['OpenAI', 'Microsoft', 'Google', 'DeepMind']
RETURN c.name as company
ORDER BY company
"""
new_companies = graph.query(new_companies_query)
for record in new_companies:
    print(f"  - {record['company']}")

print("\n추가된 인물 엔티티 확인:")
new_persons_query = """
MATCH (p:Person)
WHERE p.name IN ['Sam Altman', 'Demis Hassabis']
RETURN p.name as person, p.role as role
"""
new_persons = graph.query(new_persons_query)
for record in new_persons:
    print(f"  - {record['person']} ({record.get('role', 'CEO')})")

print("\n추가된 제품/기술 관계 확인:")
new_products_query = """
MATCH (c:Company)-[:DEVELOPS]->(p)
WHERE c.name IN ['OpenAI', 'Google', 'DeepMind']
RETURN c.name as company, p.name as product
"""
new_products = graph.query(new_products_query)
for record in new_products:
    print(f"  {record['company']} → {record['product']}")
```

</details>

### **실습 2: 커스텀 스키마로 특정 도메인 Knowledge Graph 생성**

의료 분야 뉴스를 위한 새로운 스키마를 정의하고 Knowledge Graph를 생성하세요.

**요구사항:**
- 엔티티 타입: `Disease`, `Drug`, `Researcher`, `Organization`
- 관계 타입: `TREATS`, `RESEARCHES`, `WORKS_AT`, `DEVELOPS`

In [ ]:
# 실습용 의료 뉴스
medical_news = """
화이자, 새로운 알츠하이머 치료제 개발

화이자(Pfizer)가 알츠하이머병 치료를 위한 신약 후보물질을 발표했다. 
이 신약은 베타 아밀로이드 단백질을 표적으로 하며, 임상 2상 시험에서 
긍정적인 결과를 보였다고 화이자의 연구책임자 미카엘 돌스텐(Mikael Dolsten)이 밝혔다.

존스홉킨스 대학의 신경과학 연구팀은 화이자와 협력하여 이 신약의 작용 기전을 연구하고 있으며, 
연구책임자인 제니퍼 김(Jennifer Kim) 교수는 "이 약물이 알츠하이머 진행을 늦출 수 있다"고 설명했다.
"""

# 여기에 코드를 작성하세요.
# 1. 의료 도메인용 스키마 정의
medical_entities = []
medical_relations = []
medical_potential_schema = []

# 2. 새로운 파이프라인 생성
# medical_kg_builder = SimpleKGPipeline(...)

# 3. 파이프라인 실행
# result = await medical_kg_builder.run_async(...)


<details close>
<summary>💡 정답 확인</summary>

```python
1. 의료 도메인용 스키마 정의
medical_entities = [
    "Disease",        # 질병 (알츠하이머, 파킨슨병 등)
    "Drug",           # 약물/신약
    "Researcher",     # 연구자
    "Organization"    # 조직 (회사, 대학 등)
]

medical_relations = [
    "TREATS",         # 약물 → 질병
    "RESEARCHES",     # 연구자 → 질병/약물
    "WORKS_AT",       # 연구자 → 조직
    "DEVELOPS"        # 조직 → 약물
]

medical_potential_schema = [
    ("Drug", "TREATS", "Disease"),
    ("Researcher", "RESEARCHES", "Disease"),
    ("Researcher", "RESEARCHES", "Drug"),
    ("Researcher", "WORKS_AT", "Organization"),
    ("Organization", "DEVELOPS", "Drug")
]

print("✓ 의료 도메인 스키마 정의 완료")
print(f"  - 엔티티: {medical_entities}")
print(f"  - 관계: {medical_relations}")

2. 새로운 파이프라인 생성
medical_kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    entities=medical_entities,
    relations=medical_relations,
    potential_schema=medical_potential_schema,
    text_splitter=FixedSizeSplitter(chunk_size=500, chunk_overlap=100),
    from_file=False,  # text= 로 원문을 전달하므로 False 필수 (neo4j-graphrag 1.15+)
    perform_entity_resolution=True,
    on_error="IGNORE",
    neo4j_database=NEO4J_DATABASE
)

print("✓ 의료 도메인 파이프라인 생성 완료")

3. 파이프라인 실행
async def build_medical_kg():
    """의료 도메인 Knowledge Graph 생성"""
    result = await medical_kg_builder.run_async(
        text=medical_news
    )
    return result

result = await build_medical_kg()
print("\n✓ 실습 2 완료: 의료 도메인 Knowledge Graph 생성 성공!")
print(f"  처리 결과: {result}")

생성된 의료 그래프 확인
print("\n의료 엔티티 확인:")
medical_entities_query = """
MATCH (n)
WHERE n:Disease OR n:Drug OR n:Researcher OR n:Organization
RETURN labels(n)[0] as type, n.name as name
ORDER BY type, name
"""
medical_results = graph.query(medical_entities_query)
for record in medical_results:
    print(f"  {record['type']}: {record['name']}")

print("\n의료 관계 확인:")
medical_relations_query = """
MATCH (a)-[r]->(b)
WHERE (a:Drug OR a:Researcher OR a:Organization) 
  AND type(r) IN ['TREATS', 'RESEARCHES', 'WORKS_AT', 'DEVELOPS']
RETURN a.name as from, type(r) as relation, b.name as to
"""
medical_rels = graph.query(medical_relations_query)
for record in medical_rels:
    print(f"  {record['from']} → {record['relation']} → {record['to']}")
```

</details>

### **실습 3: Cypher 쿼리로 인사이트 추출**

생성된 Knowledge Graph에서 다음 질문에 답하는 Cypher 쿼리를 작성하세요.

1. 가장 많은 관계를 가진 회사는?
2. 특정 기술(예: "GPU")을 개발하는 모든 회사는?
3. 두 회사(예: 엔비디아와 AMD) 사이의 최단 경로는?

In [ ]:
# 질문 1: 가장 많은 관계를 가진 회사
query1 = """
# TODO: Cypher 쿼리 작성
"""

# result = graph.query(query1)
# print("질문 1 결과:", result)


<details close>
<summary>💡 정답 확인</summary>

```python
질문 1: 가장 많은 관계를 가진 회사
query1 = """
MATCH (c:Company)
OPTIONAL MATCH (c)-[r]-()
WITH c, count(r) as relationCount
RETURN c.name as company, relationCount
ORDER BY relationCount DESC
LIMIT 5
"""

result = graph.query(query1)
print("질문 1 결과: 가장 많은 관계를 가진 회사")
for record in result:
    print(f"  {record['company']}: {record['relationCount']}개의 관계")
```

</details>

In [ ]:
# 질문 2: GPU를 개발하는 모든 회사
query2 = """
# TODO: Cypher 쿼리 작성
"""

# result = graph.query(query2)
# print("질문 2 결과:", result)


<details close>
<summary>💡 정답 확인</summary>

```python
질문 2: GPU를 개발하는 모든 회사
query2 = """
MATCH (c:Company)-[:DEVELOPS]->(t:Technology)
WHERE t.name CONTAINS 'GPU' OR t.name CONTAINS 'H100' OR t.name CONTAINS 'MI300'
RETURN DISTINCT c.name as company, collect(DISTINCT t.name) as technologies
ORDER BY company
"""

result = graph.query(query2)
print("질문 2 결과: GPU 관련 기술을 개발하는 회사")
for record in result:
    print(f"  {record['company']}: {', '.join(record['technologies'])}")
```

</details>

In [ ]:
# 질문 3: 엔비디아와 AMD 간 최단 경로
query3 = """
# TODO: Cypher 쿼리 작성 (shortestPath 함수 사용)
"""

# result = graph.query(query3)
# print("질문 3 결과:", result)


<details close>
<summary>💡 정답 확인</summary>

```python
질문 3: 엔비디아와 AMD 간 최단 경로
query3 = """
MATCH (nvidia:Company {name: '엔비디아'}), (amd:Company {name: 'AMD'})
MATCH path = shortestPath((nvidia)-[*..5]-(amd))
RETURN 
    [node in nodes(path) | labels(node)[0] + ': ' + node.name] as pathNodes,
    [rel in relationships(path) | type(rel)] as pathRelations,
    length(path) as pathLength
"""

result = graph.query(query3)
print("질문 3 결과: 엔비디아와 AMD 간 최단 경로")
if result:
    for record in result:
        print(f"\n경로 길이: {record['pathLength']}단계")
        print("경로:")
        for i, node in enumerate(record['pathNodes']):
            print(f"  {node}")
            if i < len(record['pathRelations']):
                print(f"    └─ [{record['pathRelations'][i]}]")
else:
    print("  두 회사 간 연결 경로를 찾을 수 없습니다.")
    print("  💡 힌트: 그래프에 두 회사가 모두 존재하고 연결되어 있는지 확인하세요.")
```

</details>

---

## **9. 정리 및 정리**

### **핵심 요약**

1. **SimpleKGPipeline**: 비정형 텍스트에서 자동으로 Knowledge Graph를 생성하는 강력한 도구
2. **스키마 정의**: 엔티티와 관계 타입을 명확히 정의하여 LLM의 추출 정확도 향상
3. **Entity Resolution**: 중복 엔티티를 자동으로 병합하여 그래프 품질 개선
4. **Lexical Graph**: Document → Chunk → Entity 구조로 출처 추적 가능
5. **다양한 LLM 지원**: OpenAI, Anthropic, Ollama 등 다양한 Provider 사용 가능

### **다음 단계**

- **GraphRAG 파이프라인**: Knowledge Graph + Vector 검색을 결합한 하이브리드 RAG
- **커스텀 파이프라인**: 개별 컴포넌트를 조합하여 특화된 파이프라인 구성
- **프로덕션 배포**: 배치 처리, 에러 핸들링, 모니터링 전략

### **참고 자료**

- [Neo4j GraphRAG Python 공식 문서](https://neo4j.com/docs/neo4j-graphrag-python/current/)
- [SimpleKGPipeline 가이드](https://neo4j.com/docs/neo4j-graphrag-python/current/kg-builder/)
- [LangChain Neo4j 통합](https://python.langchain.com/docs/integrations/graphs/neo4j_cypher/)

### **연결 종료**

In [27]:
# Neo4j 드라이버 종료
driver.close()
print("✓ Neo4j 연결 종료")

✓ Neo4j 연결 종료
